# 动手练习：研究问题、可复现分析与敏感性分析

来自《生物医学网络科学》（Biomedical Network Science）。 参考答案版。

In [ ]:
# 数据文件（JupyterLite 专用；文件已在时什么也不做）
import os, js
from pyodide.http import pyfetch
_base = str(js.location.href).split("/extensions/")[0].split("/build/")[0] + "/files/practice/data/"
os.makedirs("data", exist_ok=True)
for _f in ['breast_gds4057_expr_top1000.csv']:
    if not os.path.exists("data/" + _f):
        _r = await pyfetch(_base + _f)
        open("data/" + _f, "wb").write(await _r.bytes())


## 任务 1：阈值敏感性：枢纽排名稳定吗？

从相关矩阵建网络时要选一个网络密度（保留最强边的比例），很多论文只报告一个密度下的“枢纽节点”。本练习全部使用模拟数据（50 个节点、80 个样本，三因子模型），节点可以理解为脑区、基因或症状。

在密度 5%、10%、…、40% 下建网络，取度最高的 5 个节点为枢纽（度相同时按编号），计算每个密度下的枢纽集合与 10% 密度下枢纽集合 `ref` 的 Jaccard 相似度（交集大小 / 并集大小）。

要得到的变量：`jac`（与 `densities` 等长的列表）。

期望看到：密度离 10% 越远，枢纽集合变化越大。报告时应说明结论在哪个密度区间内成立（见 [敏感性分析](reproducible-research.sensitivity-analysis.html)）。

In [ ]:
# 准备（直接运行）
import numpy as np
import networkx as nx
import matplotlib.pyplot as plt

def simulate_corr(n_samples=80, n_nodes=50, seed=15):
    rng = np.random.default_rng(seed)
    F = rng.normal(size=(n_samples, 3))
    L = rng.uniform(0, 0.8, size=(n_nodes, 3)) * (rng.random((n_nodes, 3)) < 0.5)
    X = F @ L.T + rng.normal(size=(n_samples, n_nodes))
    return np.corrcoef(X.T)

def network_at_density(C, density):
    """保留绝对相关最强的前 density 比例的边。"""
    n = C.shape[0]
    iu = np.triu_indices(n, 1)
    keep = np.argsort(-np.abs(C[iu]), kind="stable")[:int(round(density * len(iu[0])))]
    G = nx.Graph(); G.add_nodes_from(range(n))
    G.add_edges_from(zip(iu[0][keep], iu[1][keep]))
    return G

def top_hubs(G, k=5):
    return set(sorted(G.nodes, key=lambda v: (-G.degree(v), v))[:k])

C = simulate_corr()
ref = top_hubs(network_at_density(C, 0.10))
densities = np.round(np.arange(0.05, 0.41, 0.05), 2)

In [ ]:
jac = []
for d in densities:
    h = top_hubs(network_at_density(C, d))
    jac.append(len(h & ref) / len(h | ref))
for d, j in zip(densities, jac):
    print("密度 %.2f：Jaccard = %.2f" % (d, j))
plt.figure(figsize=(4.5, 3))
plt.plot(densities, jac, "o-")
plt.xlabel("网络密度"); plt.ylabel("与 10% 密度枢纽的 Jaccard")

In [ ]:
# 核对（不用修改）
assert len(jac) == len(densities), "jac 的长度应与 densities 相同"
assert abs(jac[1] - 1.0) < 1e-9, "10% 密度与自身比较，Jaccard 应为 1"
assert all(0 <= j <= 1 for j in jac), "Jaccard 应在 [0, 1] 内；分母是并集大小，不是 5"
assert min(jac) < 1, "至少有一个密度下枢纽集合与 10% 时不同"
print("✓ 任务 1 通过")

## 任务 2：随机种子敏感性：社团划分可重复吗？

Louvain 社团检测含随机性。用 20 个种子（0–19）运行 `nx.community.louvain_communities(G, seed=s)`，记录每次的社团数、模块度 $Q$，以及与种子 0 划分的调整兰德指数（ARI，`ari` 已写好）。网络是 4 个大小不等、边界较模糊的社团（模拟数据）。

要得到的变量：`n_comms`、`Qs`、`aris`（各 20 个值）。

期望看到：$Q$ 的波动很小，但社团数和具体划分可能变化。报告时应给出多次运行的范围或共识划分（见 [随机性控制](reproducible-research.randomness-robustness.html)）。

In [ ]:
# 准备（直接运行）
def ari(a, b):
    """调整兰德指数：1 表示完全一致，约 0 表示与随机划分相当。"""
    a = np.unique(a, return_inverse=True)[1]; b = np.unique(b, return_inverse=True)[1]
    M = np.zeros((a.max() + 1, b.max() + 1))
    np.add.at(M, (a, b), 1)
    c2 = lambda x: x * (x - 1) / 2
    s_ij, s_a, s_b, n2 = c2(M).sum(), c2(M.sum(1)).sum(), c2(M.sum(0)).sum(), c2(len(a))
    exp = s_a * s_b / n2
    return (s_ij - exp) / (0.5 * (s_a + s_b) - exp)

def labels_from(comms, n):
    lab = np.zeros(n, dtype=int)
    for i, c in enumerate(comms):
        lab[list(c)] = i
    return lab

G2 = nx.random_partition_graph([30, 25, 25, 20], 0.25, 0.04, seed=15)
n2 = G2.number_of_nodes()
base = labels_from(nx.community.louvain_communities(G2, seed=0), n2)

In [ ]:
n_comms, Qs, aris = [], [], []
for s in range(20):
    comms = nx.community.louvain_communities(G2, seed=s)
    n_comms.append(len(comms))
    Qs.append(nx.community.modularity(G2, comms))
    aris.append(ari(labels_from(comms, n2), base))
print("社团数 %d–%d；Q %.3f–%.3f；与种子 0 的 ARI %.2f–%.2f"
      % (min(n_comms), max(n_comms), min(Qs), max(Qs), min(aris), max(aris)))

In [ ]:
# 核对（不用修改）
assert len(n_comms) == len(Qs) == len(aris) == 20, "每个种子各记录一个值，共 20 个"
assert abs(aris[0] - 1.0) < 1e-9, "种子 0 与自身比较，ARI 应为 1"
assert max(Qs) - min(Qs) < 0.1, "Q 的波动应很小；检查是否用 nx.community.modularity(G2, comms)"
assert all(-0.5 <= a <= 1 + 1e-9 for a in aris), "ARI 应不超过 1"
print("✓ 任务 2 通过")

## 任务 3：替代解释：组间差异是“结构不同”，还是“连接整体变弱”？

两组的潜在结构完全相同，只是 B 组的测量噪声更大（类似头动或批次差异）。先用固定阈值 $|r|>0.3$ 比较两组的平均聚集系数，再改为两组都取相同密度（15%）比较。

要得到的变量：`cc_fixed`、`cc_matched`（都是 `{"A": 值, "B": 值}` 的字典）。

期望看到：固定阈值下两组差异很大；密度匹配后差异明显缩小（见 [替代解释排查表](reproducible-research.sensitivity-analysis.html)）。
思考：既然真实结构完全相同，密度匹配后剩下的差异从哪里来？

In [ ]:
# 准备（直接运行）
def group_corr(noise, n_samples=80, n_nodes=50, seed=0):
    rng = np.random.default_rng(1000 + seed)
    struct = np.random.default_rng(15)                 # 两组共享同一潜在结构
    L = struct.uniform(0, 0.8, size=(n_nodes, 3)) * (struct.random((n_nodes, 3)) < 0.5)
    F = rng.normal(size=(n_samples, 3))
    X = F @ L.T + noise * rng.normal(size=(n_samples, n_nodes))
    return np.corrcoef(X.T)

def net_threshold(C, r):
    A = (np.abs(C) > r).astype(int); np.fill_diagonal(A, 0)
    return nx.from_numpy_array(A)

CA, CB = group_corr(noise=1.0, seed=1), group_corr(noise=1.6, seed=2)

In [ ]:
cc_fixed = {k: nx.average_clustering(net_threshold(C_, 0.3)) for k, C_ in [("A", CA), ("B", CB)]}
cc_matched = {k: nx.average_clustering(network_at_density(C_, 0.15)) for k, C_ in [("A", CA), ("B", CB)]}
for k, C_ in [("A", CA), ("B", CB)]:
    print("%s 组：固定阈值密度 %.3f、聚集系数 %.3f；密度 15%% 时聚集系数 %.3f"
          % (k, nx.density(net_threshold(C_, 0.3)), cc_fixed[k], cc_matched[k]))

In [ ]:
# 核对（不用修改）
assert None not in cc_fixed.values() and None not in cc_matched.values(), "两个字典都要填上 A、B 两组的值"
gap_fixed = abs(cc_fixed["A"] - cc_fixed["B"])
gap_matched = abs(cc_matched["A"] - cc_matched["B"])
assert gap_fixed > gap_matched, "密度匹配后两组差异应缩小；检查 cc_matched 是否用了 network_at_density(C, 0.15)"
assert abs(nx.density(network_at_density(CA, 0.15)) - 0.15) < 0.01, "密度匹配网络的密度应约为 0.15"
print("✓ 任务 3 通过")

## 任务 4：蒙特卡洛误差：置换次数够不够？

写函数 `emp_p(obs, null)`，按 $\hat p=(b+1)/(B+1)$ 计算经验 $p$ 值，$b$ 为零分布中不小于观测值的个数。然后对同一组数据（两组各 30 人、组间差异处在边界上）做均值差的置换检验：分别用 $B=99$ 与 $B=999$，各换 30 个种子（`seed = 0..29`），比较 $\hat p$ 在种子之间的标准差。

要得到的变量：函数 `emp_p`，`p_small`、`p_large`（各 30 个 $\hat p$）。

期望看到：$B=99$ 时 $\hat p$ 的标准差明显更大，不同种子可能落在 0.05 的两侧；理论上标准误约为 $\sqrt{p(1-p)/B}$（见 [随机性控制](reproducible-research.randomness-robustness.html)）。

In [ ]:
# 准备（直接运行）
rng4 = np.random.default_rng(44)
x1 = rng4.normal(0.0, 1, 30)
x2 = rng4.normal(0.5, 1, 30)
pooled = np.concatenate([x1, x2])
obs_diff = abs(x2.mean() - x1.mean())

def perm_null(B, seed):
    r = np.random.default_rng(seed)
    out = np.empty(B)
    for i in range(B):
        s = r.permutation(pooled)
        out[i] = abs(s[30:].mean() - s[:30].mean())
    return out

In [ ]:
def emp_p(obs, null):
    null = np.asarray(null)
    return (np.sum(null >= obs) + 1) / (len(null) + 1)

p_small = [emp_p(obs_diff, perm_null(99, s)) for s in range(30)]
p_large = [emp_p(obs_diff, perm_null(999, s)) for s in range(30)]
print("B = 99：p 均值 %.3f，标准差 %.4f，p < 0.05 的比例 %.2f" % (np.mean(p_small), np.std(p_small), np.mean(np.array(p_small) < 0.05)))
print("B = 999：p 均值 %.3f，标准差 %.4f，p < 0.05 的比例 %.2f" % (np.mean(p_large), np.std(p_large), np.mean(np.array(p_large) < 0.05)))

In [ ]:
# 核对（不用修改）
assert emp_p(5, [1, 2, 3]) == 0.25, "零分布中没有值 ≥ 观测值时，p = 1/(B+1)，不应为 0"
assert emp_p(2, [1, 2, 3]) == 0.75, "应统计“不小于”观测值的个数（含相等），再按 (b+1)/(B+1) 计算"
assert len(p_small) == len(p_large) == 30, "两种 B 各 30 个种子"
assert np.std(p_small) > np.std(p_large), "B = 99 时 p 的种子间波动应更大"
assert min(p_small) >= 1 / 100, "B = 99 时 p 的最小可能值是 0.01"
print("✓ 任务 4 通过")

## 任务 5：循环分析：先挑最大差异再检验

模拟 200 次“没有任何组间差异”的研究：每次两组各 20 人、300 条边。做法甲：在全部数据上挑出 $|t|$ 最大的 10 条边，再在**同一数据**上做 $t$ 检验，看这 10 条边中 $p<0.05$ 的比例。做法乙：一半被试（每组前 10 人）用来挑边，另一半（后 10 人）用来检验。

要得到的变量：`frac_circular`、`frac_split`（200 次研究、每次 10 条边的平均“显著”比例）。

期望看到：做法甲几乎全部“显著”，做法乙回到名义水平 5% 左右（见 [循环分析](reproducible-research.sensitivity-analysis.html)）。

> **提示：** `scipy.stats.ttest_ind(a, b, axis=0)` 可以一次检验所有边。

In [ ]:
# 准备（直接运行）
from scipy import stats
rng5 = np.random.default_rng(55)
studies = [(rng5.normal(size=(20, 300)), rng5.normal(size=(20, 300))) for _ in range(200)]

In [ ]:
fc, fs = [], []
for g1, g2 in studies:
    t_all, p_all = stats.ttest_ind(g1, g2, axis=0)
    top = np.argsort(-np.abs(t_all))[:10]
    fc.append(np.mean(p_all[top] < 0.05))
    t_sel, _ = stats.ttest_ind(g1[:10], g2[:10], axis=0)
    top2 = np.argsort(-np.abs(t_sel))[:10]
    _, p_test = stats.ttest_ind(g1[10:, top2], g2[10:, top2], axis=0)
    fs.append(np.mean(p_test < 0.05))
frac_circular, frac_split = float(np.mean(fc)), float(np.mean(fs))
print("做法甲（同一数据选择并检验）：%.2f；做法乙（分半）：%.3f" % (frac_circular, frac_split))

In [ ]:
# 核对（不用修改）
assert frac_circular is not None and frac_circular > 0.8, "同一数据选择并检验时，几乎所有被选边都会“显著”；检查是否在全部数据上选择和检验"
assert frac_split is not None and frac_split < 0.15, "分半后“显著”比例应接近 5%；检查检验是否只用了后 10 人、且只检验前半挑出的边"
print("✓ 任务 5 通过")

## 任务 6：开放小分析：ESR1 是不是共表达网络的枢纽？一个小型多元宇宙

数据：`data/breast_gds4057_expr_top1000.csv`（1000 个高变基因 × 103 例乳腺癌活检，GEO GDS4057，来源见 [基因网络推断的真实数据案例](gene-network-inference.html)）。

研究问题：雌激素受体基因 *ESR1* 是不是这个共表达网络的枢纽？至少组合两个分析维度，例如相关类型（皮尔逊/斯皮尔曼）× 网络密度（如 2%、5%、10%）× 基因集合（方差最大的 500 个/全部 1000 个），在每个宇宙中计算 *ESR1* 的度在全部基因中的排名（1 为最高）。

要得到的变量：`results`（列表，每个宇宙一个字典，至少包含键 `"rank"` 与 `"n_genes"`，至少 6 个宇宙）。

思考：结论“ESR1 是枢纽”在多少个宇宙中成立？如果要写进论文，怎样表述才诚实？可以用 [多元宇宙分析](reproducible-research.multiverse-analysis.html) 中的设定曲线展示。

In [ ]:
# 准备（直接运行）
import pandas as pd
from itertools import product
expr = pd.read_csv("data/breast_gds4057_expr_top1000.csv", index_col="gene")
Xb = expr.T
print(Xb.shape, "ESR1" in Xb.columns)

In [ ]:
results = []
var_order = Xb.var().sort_values(ascending=False).index
for method, density, ng in product(["pearson", "spearman"], [0.02, 0.05, 0.10], [500, 1000]):
    genes_u = list(var_order[:ng])
    if "ESR1" not in genes_u:
        genes_u = genes_u[:-1] + ["ESR1"]
    Cm = Xb[genes_u].corr(method=method).values
    Gm = network_at_density(Cm, density)
    deg = np.array([Gm.degree(i) for i in range(len(genes_u))])
    i_esr1 = genes_u.index("ESR1")
    rank = int(1 + np.sum(deg > deg[i_esr1]))
    results.append({"相关": method, "密度": density, "n_genes": ng, "rank": rank, "前 5%": rank <= 0.05 * ng})
res_df = pd.DataFrame(results)
print(res_df.to_string(index=False))

In [ ]:
# 核对（不用修改）
assert results is not None and len(results) >= 6, "至少要有 6 个宇宙（例如 2 种相关 × 3 种密度）"
assert all("rank" in r and "n_genes" in r for r in results), "每个宇宙的字典都要有 rank 与 n_genes"
assert all(1 <= r["rank"] <= r["n_genes"] for r in results), "排名应在 1 到基因数之间"
print("✓ 任务 6 通过")